# Home Assignment 3 - CS5720 Neural Network and Deep Learning
**Student:** PUPPALA CHARAN SAI  
**Student ID:** 700787150  
**University:** University of Central Missouri  

## Part II, Question 1: Manual 2-D Convolution

In [2]:
import numpy as np

image = np.array([[1,1,1,0,0],
                  [0,1,1,1,0],
                  [0,0,1,1,1],
                  [0,0,1,1,0],
                  [0,1,1,0,0]])
filter_kernel = np.array([[1,0,1],
                          [0,1,0],
                          [1,0,1]])

def apply_filter(image, kernel, stride=1):
    # Valid CNN-style filtering (cross-correlation); this symmetric kernel
    # produces the same result if flipped for mathematical convolution.
    height = (image.shape[0] - kernel.shape[0]) // stride + 1
    width = (image.shape[1] - kernel.shape[1]) // stride + 1
    result = np.zeros((height, width), dtype=int)
    for i in range(height):
        for j in range(width):
            r, c = i * stride, j * stride
            patch = image[r:r + kernel.shape[0], c:c + kernel.shape[1]]
            result[i, j] = np.sum(patch * kernel)
    return result

output = apply_filter(image, filter_kernel, stride=1)
print('Output feature map (stride=1):\n', output)
print('Output shape:', output.shape)
assert output.tolist() == [[4,3,4],[2,4,3],[2,3,4]]

output_stride2 = apply_filter(image, filter_kernel, stride=2)
print('\nOutput feature map (stride=2):\n', output_stride2)
print('Output shape:', output_stride2.shape)


Output feature map (stride=1):
 [[4 3 4]
 [2 4 3]
 [2 3 4]]
Output shape: (3, 3)

Output feature map (stride=2):
 [[4 4]
 [2 4]]
Output shape: (2, 2)


With stride 2, the filter moves two cells per step, giving a 2 x 2 output instead of 3 x 3.

## Part II, Question 2: Pretrained ResNet18 - Cats vs Dogs

In [3]:
import time
import copy
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5
BATCH_SIZE = 32
print('Device:', DEVICE)


  Using cached torchvision-0.23.0-cp39-cp39-win_amd64.whl (1.6 MB)
Note: you may need to restart the kernel to use updated packages.


E:\Users\KIDDO\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
E:\Users\KIDDO\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Device: cpu


In [4]:
# Two-class CIFAR-10: cat (label 3) vs dog (label 5).
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
train_raw = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_raw = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

class CatDogDataset(Dataset):
    def __init__(self, original):
        self.original = original
        self.indices = [i for i, label in enumerate(original.targets) if label in (3, 5)]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.original[self.indices[idx]]
        return image, (0 if label == 3 else 1)

train_loader = DataLoader(CatDogDataset(train_raw), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)
test_loader = DataLoader(CatDogDataset(test_raw), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)
print('Train images:', len(train_loader.dataset), 'Test images:', len(test_loader.dataset))


100%|████████████████████████████████████████████████████████████████████████████████| 170M/170M [16:26<00:00, 173kB/s]


Train images: 10000 Test images: 2000


In [5]:
# Download ImageNet pretrained ResNet18 weights once, then start both
# experiments from identical initial weights, including the new classifier.
base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
base.fc = nn.Linear(base.fc.in_features, 2)
initial_weights = copy.deepcopy(base.state_dict())

def build_model(fine_tune=False):
    net = models.resnet18(weights=None)
    net.fc = nn.Linear(net.fc.in_features, 2)
    net.load_state_dict(initial_weights)
    for parameter in net.parameters():
        parameter.requires_grad = False
    for parameter in net.fc.parameters():
        parameter.requires_grad = True
    if fine_tune:
        for parameter in net.layer4.parameters():
            parameter.requires_grad = True
    return net.to(DEVICE)

def train_and_evaluate(fine_tune=False):
    net = build_model(fine_tune)
    n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam((p for p in net.parameters() if p.requires_grad), lr=1e-4)
    epoch_losses = []
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for epoch in range(EPOCHS):
        net.train()
        # Freeze running statistics of BatchNorm, including in the last block.
        for layer in net.modules():
            if isinstance(layer, nn.BatchNorm2d):
                layer.eval()
        running_loss, seen = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = net(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            seen += images.size(0)
        avg_loss = running_loss / seen
        epoch_losses.append(avg_loss)
        print(f"{'Fine-tuned' if fine_tune else 'Frozen'} | Epoch {epoch+1}/{EPOCHS} | Loss {avg_loss:.4f}")
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            predicted = net(images).argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return {'Trainable Parameters': n_params,
            'Training Time (s)': round(elapsed, 2),
            'Accuracy (%)': round(100 * correct / total, 2),
            'Losses': epoch_losses}


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\KIDDO/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 57.6MB/s]


In [ ]:
# Run both methods for exactly the same number of epochs.
frozen = train_and_evaluate(fine_tune=False)
fine_tuned = train_and_evaluate(fine_tune=True)

comparison = pd.DataFrame([
    {'Method': 'Frozen Feature Extractor', **{k: v for k, v in frozen.items() if k != 'Losses'}},
    {'Method': 'Fine-Tuned Network', **{k: v for k, v in fine_tuned.items() if k != 'Losses'}}
])
display(comparison)
comparison.to_csv('experiment_results.csv', index=False)

plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS+1), frozen['Losses'], marker='o', label='Frozen Feature Extractor')
plt.plot(range(1, EPOCHS+1), fine_tuned['Losses'], marker='s', label='Fine-Tuned Network')
plt.xlabel('Epoch')
plt.ylabel('Average Training Loss')
plt.title('ResNet18: Feature Extraction vs Fine-Tuning')
plt.xticks(range(1, EPOCHS+1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=160)
plt.show()


### Discussion
The frozen feature extractor updates only the new classifier and usually trains faster. Fine-tuning also updates the last convolutional block, so it requires more computation. The pretrained features can already be useful for cats and dogs. Fine-tuning can adapt those features, but may not always improve test accuracy. Compare the measured times and accuracies in the table above. Compare the loss curves to see which approach learned more quickly. The preferred method depends on accuracy, time, and available computing resources.